# Notebook to train SPA-C's model

**@author**: Alexis Mergez  
**@Last Modified**: 26/01/12

# Librairies

In [ ]:
from torch.utils.data import DataLoader

import os, time, datetime

import sys
sys.path.append('../py_scripts')

from SPAC_Dataset import dataset
from models import *

# Variables

In [ ]:
input_shape = (20, 20)
bin_size = 5000
latent_width = 64
batch_size = 256
n_epochs = 10
n_workers = 12

# CuDNN optimisation
torch.backends.cudnn.benchmark = True
main_dir = "/home/amergez/SPA-C/models"
data_dir = "/home/amergez/SPA-C/datasets/CHM13.DS/CHM13v2.DS_5K10S_Q0.C4X.hdf5"
thres_dir = "/home/amergez/SPA-C/datasets/CHM13.DS/CHM13v2.DS_5K10S_Q0.C4X_THRES.hdf5"
hg002_dir = "/home/amergez/SPA-C/datasets/HG002.DS/HG002.DS_5K10S_Q0.hdf5"
model_name = "FF.401"

model_dir = os.path.join(main_dir, model_name)
os.makedirs(model_dir, exist_ok=True)

print("--- Directories ---")
print("Main :", main_dir)
print("Data :", data_dir)
print("Model :", model_dir)

# Get cpu, gpu or mps device for training.
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

# Loading and preparing dataset

In [ ]:
# Loading an existing dataset
print("Preparing train dataset ...")
ds = dataset(data_dir, image_size = 10, bin_size = bin_size, skip_whole=True, skip_check=True)

# MinMax normalisation
ds.minmax(max_val=31.0) # Similar to AutoHic normalisation

In [ ]:
import torchvision.transforms as transforms

class RandomBorderMasking(torch.nn.Module):
    """
    Applying a random mask
    Args:
        p (float): Probability.
    """
    def __init__(self, p=1.0, padding=0):
        super().__init__()
        self.p = p
        self.padding = padding

    def forward(self, x):
        if torch.rand(1).item() > self.p or torch.isin(-1, x):
            return x 
        
        # Random sized mask (1-7)
        k = torch.randint(1, 8, (1,)).item()

        # Choice of the side (0 : Start, 1: End)
        border_choice = torch.randint(0, 2, (1,)).item()

        # Applying the mask
        if border_choice == 0:  # Start
            x[:,0:k,:] = self.padding
            x[:,:,0:k] = self.padding
        elif border_choice == 1:  # End
            x[:,20-k:,:] = self.padding
            x[:,:,20-k:] = self.padding
        return x

    def __repr__(self):
        return f"RandomBorderMasking(p={self.p})"

transform = transforms.Compose([
    RandomBorderMasking(p=0.5, padding=-1),
])

In [ ]:
ds.summary()
ds.split_by_group()
ds.summary()

# Converting it to Torch dataloader
train_dataloader = DataLoader(ds.get_as_torch("train", transform=transform), batch_size=batch_size, shuffle=True, num_workers=n_workers, pin_memory=True)
val_dataloader = DataLoader(ds.get_as_torch("val", transform=transform), batch_size=batch_size, shuffle=False, num_workers=n_workers, pin_memory=True)
test_dataloader = DataLoader(ds.get_as_torch("test", transform=None), batch_size=batch_size, shuffle=False, num_workers=n_workers, pin_memory=True)

# Creating and training the model

In [ ]:
# Creating the model instance
model = SPAC(
    device=device,
    name=model_name,
    latent_width=latent_width,
    input_shape=(1, 20, 20)
)

In [ ]:
# Viz
from torchinfo import summary

print(summary(
            model.model,
            input_size=(512, 1, 20, 20),
            col_names=["num_params", "input_size", "output_size"]
        ))

## Training

In [ ]:
# Training
model.training(
    dataLoader=train_dataloader,
    val_dataLoader=val_dataloader,
    savedir=model_dir,
    nepochs=30,
    freeze_encoder=False,
    FocalLoss_alpha=None, # no class balancing
    FocalLoss_gamma=1,
    FocalLoss_pos_weight=1.0,
    FocalLoss_neg_weight=1.2,
    add_scheduler=True,
    lr=0.001 #0.001 by default
)

## Loading weights

In [ ]:
best_weights = 4
model.load_weights(os.path.join(model_dir, f"{model_name}.Classifier.E{best_weights:>02}.pth"))

## Setting the threshold

In [ ]:
thres_ds = dataset(thres_dir, image_size = 10, bin_size = bin_size, skip_whole=True, skip_check=True)

# MinMax normalisation
thres_ds.minmax(max_val=31.0) # Similar to AutoHic normalisation

thres_ds.split_by_group()
thres_ds.summary()

thresval_dataloader = DataLoader(thres_ds.get_as_torch("val", transform=None), batch_size=batch_size, shuffle=False, num_workers=n_workers, pin_memory=True)

In [ ]:
thres_ds.val["names"]

In [ ]:
# Predicting on validation set
val_probs, _ = model.predict(
    thresval_dataloader,
    savedir = os.path.join(main_dir, model_name, f"chim_scores_CHM13_val.json"),
    names=thres_ds.val["names"]
)

In [ ]:
with open(os.path.join(main_dir, model_name, f"chim_scores_CHM13_val.json"), "r") as handle:
    val_probs = json.load(handle)

# Removing probabilities of inter matrices
val_probs = {key: value for key, value in val_probs.items() if len(key.split("_C")) != 1}

In [ ]:
val_probs.keys()

In [ ]:
# Computing ranges
intra_probs = {}

for key, value in val_probs.items():
    # Intra contig prediction
    # Decomposing the key
    chrom_name, bin_number, corner_id = key.split("|Val")[0].rsplit("_", 2)
    bin_number = int(bin_number)

    # Adding chromosome to intra_probs if not already
    if not chrom_name in intra_probs.keys():
        intra_probs[chrom_name] = {}

    # Adding bin_number if not already
    if not bin_number in intra_probs[chrom_name].keys():
        intra_probs[chrom_name][bin_number] = {"C2": None, "C3": None}

    # Adding the value
    intra_probs[chrom_name][bin_number][corner_id] = value

In [ ]:
# Sorted indices for each sequence
indices = {
    key: sorted(list(value.keys()))
    for key, value in intra_probs.items()
}

In [ ]:
#% Copied from Chimera_predictor.py v0.11.0.
def mean_predictions(x, ctg_probs):
    """
    Smoothing 'algorithm' using np.convolve to compute the rolling average with a size of <smoothing window>.
    If the contig is smaller than the smoothing window, only the mean is computed.
    """

    mean_probs = np.mean([
        [ctg_probs[k]["C2"] for k in x],
        [ctg_probs[k]["C3"] for k in x]
    ], axis=0)

    return mean_probs

def compute_range(x, probs, threshold):
    """
    Two cases :
    - The size of the chimeric region is greater than the minimal prediction size for scaffolding -> Considered as a new contig.
    - Else -> We take the mean position as breakpoint for the whole contig.
    """

    in_keep = False
    current_range = None
    current_bad_range = None
    current_part_id = 1
    _ctg_parts = {}

    for bin_id, pred in zip(x, probs):

        if (pred >= threshold):  # We keep this bin

            if in_keep == False:  # We were not in a keeping zone hence we create one
                current_range = [bin_id, bin_id]
                in_keep = True

                if current_bad_range is not None:  # We were in a bad range
                    # Updating the last bin
                    current_bad_range[1] = bin_id

                    # Updating the contig partition
                    _ctg_parts[current_part_id] = {
                        "chimeric": True,
                        "range": [current_bad_range[0], current_bad_range[1]]
                    }

                    current_part_id += 1
                    current_bad_range = None

            else:  # We were already in a keeping zone, so we update the last element of the list
                current_range[-1] = bin_id

        if (pred < threshold) or bin_id == max(
                x):  # We exit the keeping region or we are at the end anyway
            in_keep = False

            if current_range is not None:  # We were in a keeping zone, so we exit it
                # Updating last bin_id to the range
                current_range[1] = bin_id

                # Updating the contig partition
                _ctg_parts[current_part_id] = {
                    "chimeric": False,
                    "range": [current_range[0], current_range[1]]
                }

                current_part_id += 1
                current_range = None

            if current_bad_range is None:
                current_bad_range = [bin_id, bin_id]
            else:
                current_bad_range[-1] = bin_id

    return _ctg_parts

In [ ]:
m_probs = {}
res = {} 
ctg_parts = {}
chrom_ranges = {} # Stores the start and end bin of each chromosome/contig. They are left and right shifted.
#thres=0.05

cuts = {}
for threshold in tqdm(range(0,100,5), desc="Testing all thresholds...", unit="threshold"):
    thres = threshold/100
    res[thres] = {"Chimeric": {}, "Correct": {}}
    
    for seq_name in tqdm(sorted(list(intra_probs.keys())), desc=f"Processing validation sequences...", unit="ctg"):
        # Adding current contig entry into ctg_parts
        ctg_parts[seq_name] = []
        cuts[seq_name] = []
        # Ordering bin number for the given chromosome
        x = indices[seq_name]
        chrom_ranges[seq_name] = (x[0], x[-1]) # Left and right shifted
    
        # Computing mean probability curve
        mean_probs = mean_predictions(x, intra_probs[seq_name])
    
        # Storing smoothed probabilities
        m_probs[seq_name] = {
            "x": x,
            "probs": mean_probs
        }
    
        ctg_parts[seq_name] = compute_range(x, mean_probs, threshold=thres)
    
        # Processing ctg_parts
        for region in ctg_parts[seq_name].values():
            if region['chimeric']:
                length = region['range'][1]-region['range'][0]
                if length<20:
                    cuts[seq_name].append((region['range'][1]+region['range'][0])//2)
                else:
                    cuts[seq_name].extend(region['range'])
    
        # Processing cuts into distance from breakpoint and number of false cuts
        if "|" in seq_name:
            cut_type = "Chimeric"
        else:
            cut_type = "Correct"
    
        res[thres][cut_type][seq_name] = [] # Creating entry
        
        for cut_pos in cuts[seq_name]:
            if cut_type == "Chimeric":
                res[thres][cut_type][seq_name].append(abs(cut_pos-20)) # Adding the distance to the breakpoint
            else:
                res[thres][cut_type][seq_name].append(cut_pos) # The number of cuts is more important than the actual position

res

In [ ]:
# Processing results
## Cumulative sum of error distance and number of cuts in chromosomes
df = {}
for thres, cuts_pos in res.items():
    chim, correct = [], 0
    for dists in cuts_pos["Chimeric"].values():
        chim.extend(dists)
    for pos in cuts_pos["Correct"].values():
        correct += len(pos)
    df[thres] = {"Cumul_Dist": np.sum(chim), "N_miscuts": correct}

import pandas as pd
df = pd.DataFrame.from_dict(
    df, orient="index"
).rename_axis(
    "Threshold"
).reset_index().sort_values("Threshold")
df

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))

sns.lineplot(
    data=df, x="Threshold", y="Cumul_Dist",
    marker="o", estimator=None, sort=True, ax=ax1
)
ax1.set_ylabel(f"Cumulative distance from breakpoint (n={len(res[0.0]['Chimeric'].keys())})")
ax1.set_title("Distance from breakpoint and number of cuts in chromosomes")

ax2 = ax1.twinx()
sns.lineplot(
    data=df, x="Threshold", y="N_miscuts",
    marker="o", estimator=None, sort=True, ax=ax2,
    color="tab:orange"
)
ax2.set_ylabel("Number of incorrect cuts in chromosomes")
ax2.set_xlabel("Threshold")

ax1.grid(True, alpha=0.3)

fig.tight_layout()

In [ ]:
with open(os.path.join(main_dir, f"ctgs_parts.json"), "w") as handle:
    json.dump(ctg_parts, handle, indent=2)

## Testing on the test dataset

In [ ]:
model.eval(
    test_dataloader, 
    threshold=.2, # Given the best threshold fixed in the previous section 
    savedir=os.path.join(model_dir, f"{model_name}_E{best_weights:>02}_CHM13.DS.test_scores.png"),
    names=ds.test["names"],
    #title=f"{model_name} - CHM13.DS.test - E{best_weights:>02} - Thres: 0.05"
)

# Predicting on HG002.DS

In [ ]:
HG002 = dataset(
    hg002_dir,
    bin_size=5000,
    image_size=10,
    skip_check=True
)
HG002.minmax(max_val=47.0)
HG002.summary()

hg002_dataloader = DataLoader(HG002.get_as_torch(), batch_size=batch_size, shuffle=False, num_workers=n_workers, pin_memory=True)

In [ ]:
model.eval(
    hg002_dataloader, 
    threshold=.05, 
    savedir=os.path.join(model_dir, f"{model_name}_E{best_weights:>02}_HG002.DS_scores.png"),
    names=HG002.dataset["names"],
    #title=f"{model_name} - HG002.DS - E{best_weights:>02} - Thres: 0.05"
)